# Advanced Retrieval Strategies

This file teaches you retrieval techniques that transform the query or use multi-step retrieval.
By the end, you will know how to use multi-query, HyDE, sub-question decomposition, self-query, parent-child retrieval, and contextual compression.

## Setup

In [1]:
!pip install python_dotenv


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

api_key = os.getenv("API_KEY")

In [3]:
import json
import numpy as np
from openai import OpenAI

client = OpenAI(api_key = api_key)

In [4]:
EMBED_MODEL = "text-embedding-3-small"
MODEL = "gpt-4o-mini"

In [5]:
def get_embedding(text):
    response = client.embeddings.create(
        model=EMBED_MODEL, input=[text]
    )
    embedding = response.data[0].embedding
    return embedding

def get_embeddings(texts):
    response = client.embeddings.create(
        model=EMBED_MODEL, input=texts
    )
    embeddings = []
    for item in response.data:
        embeddings.append(item.embedding)
    return embeddings

In [6]:
def cosine_similarity(a, b):
    a = np.array(a)
    b = np.array(b)
    dot_product = np.dot(a, b)
    norm_a = np.linalg.norm(a)
    norm_b = np.linalg.norm(b)
    similarity = dot_product / (norm_a * norm_b)
    return float(similarity)

**What happened:** We loaded libraries and created helper functions for embeddings and similarity.

### Knowledge Base

In [7]:
KNOWLEDGE_BASE = [
    {"id": 0,
     "text": "HPA scales Pods based on CPU. Default interval is 15 seconds. Supports custom metrics.",
     "source": "scaling", "date": "2024-01"},
    {"id": 1,
     "text": "VPA adjusts CPU and memory requests. Changes resource requests, not replica count.",
     "source": "scaling", "date": "2024-01"},
    {"id": 2,
     "text": "KEDA scales based on external events like queues. Supports 50+ sources. Scales to zero.",
     "source": "scaling", "date": "2024-03"},
]

In [8]:
KNOWLEDGE_BASE += [
    {"id": 3,
     "text": "Kubernetes Services: ClusterIP for internal, NodePort for external, LoadBalancer for cloud.",
     "source": "networking", "date": "2024-02"},
    {"id": 4,
     "text": "Network Policies restrict Pod-to-Pod traffic using label selectors. Default: all allowed.",
     "source": "security", "date": "2024-02"},
    {"id": 5,
     "text": "RBAC uses Roles and ClusterRoles to define permissions with RoleBindings.",
     "source": "security", "date": "2024-01"},
]

In [9]:
KNOWLEDGE_BASE += [
    {"id": 6,
     "text": "Helm charts template Kubernetes manifests. Supports dependencies, hooks, rollbacks.",
     "source": "tooling", "date": "2024-03"},
    {"id": 7,
     "text": "Istio provides mutual TLS, traffic splitting, circuit breaking, tracing with Jaeger.",
     "source": "mesh", "date": "2024-04"},
    {"id": 8,
     "text": "Prometheus scrapes /metrics endpoints. Grafana visualizes. AlertManager routes alerts.",
     "source": "monitoring", "date": "2024-02"},
    {"id": 9,
     "text": "PersistentVolumes provide storage. StorageClasses enable dynamic provisioning with CSI.",
     "source": "storage", "date": "2024-01"},
]

In [10]:
doc_texts = []
for doc in KNOWLEDGE_BASE:
    doc_texts.append(doc["text"])

DOC_EMBEDDINGS = get_embeddings(doc_texts)
DOC_EMBEDDINGS = np.array(DOC_EMBEDDINGS, dtype="float32")

In [11]:
def simple_search(query_embedding, top_k=3):
    
    scores = []
    for i in range(len(KNOWLEDGE_BASE)):
        score = cosine_similarity(
            query_embedding, 
            DOC_EMBEDDINGS[i]
        )
        scores.append((i, score))
    scores.sort(key=lambda x: x[1], reverse=True)

    results = []
    for i, score in scores[:top_k]:
        results.append((KNOWLEDGE_BASE[i], score))
    return results

**What happened:** We set up a knowledge base of 10 Kubernetes documents and pre-computed all embeddings.

## Multi-Query Retrieval

**What it does:** Generates multiple versions of the query and searches with each one. Combines all results.

**When to use it:** When user queries are short or ambiguous. Different phrasings can find different relevant documents.

**Steps:**
1. Ask the LLM to rephrase the query 3 different ways.
2. Search with each version.
3. Combine and deduplicate the results.

In [12]:
def multi_query_search(query):
    response = client.chat.completions.create(
        model=MODEL, 
        max_tokens=200, 
        temperature=0.7,
        response_format={
            "type": "json_object"
        },
        messages=[
            {
                "role": "system", 
                "content": "Return JSON only."
            },
            {
                "role": "user", 
                "content":
                "Generate 3 different versions of this "
                + f"search query.\nQuery: \"{query}\"\n"
                + 'Return: {"queries": ["q1","q2","q3"]}'
            }
        ]
    )
    
    content = response.choices[0].message.content
    
    queries = json.loads(content)["queries"]
    queries.append(query)
    
    return queries

In [ ]:
query = "How do I make my app scale automatically?"

queries = multi_query_search(query)
print("multi_query_search queries\n", queries)



# Search with all variations and combine
all_results = {}

for query in queries:
    
    embedding = get_embedding(query)
    results = simple_search(embedding, top_k=3)

    for i in range(len(results)):
        doc = results[i][0]
        score = results[i][1]
        
        doc_id = doc["id"]
        
        if doc_id not in all_results:
            all_results[doc_id] = (doc, score)
        elif score > all_results[doc_id][1]:
            all_results[doc_id] = (doc, score)

print("all_results: \n", all_results)

ranked = sorted(
    all_results.values(),
    key=lambda x: x[1], reverse=True
)

print("ranked\n", ranked)

multi_query_search queries
 ['What are the steps to enable automatic scaling for my app?', 'How can I implement automatic scaling for my application?', 'What methods exist for making my app scale on its own?', 'How do I make my app scale automatically?']
all_results: 
 {2: ({'id': 2, 'text': 'KEDA scales based on external events like queues. Supports 50+ sources. Scales to zero.', 'source': 'scaling', 'date': '2024-03'}, 0.4280880980644334), 0: ({'id': 0, 'text': 'HPA scales Pods based on CPU. Default interval is 15 seconds. Supports custom metrics.', 'source': 'scaling', 'date': '2024-01'}, 0.3533963285055371), 1: ({'id': 1, 'text': 'VPA adjusts CPU and memory requests. Changes resource requests, not replica count.', 'source': 'scaling', 'date': '2024-01'}, 0.3180883555126052)}
ranked
 [({'id': 2, 'text': 'KEDA scales based on external events like queues. Supports 50+ sources. Scales to zero.', 'source': 'scaling', 'date': '2024-03'}, 0.4280880980644334), ({'id': 0, 'text': 'HPA scale

**What happened:** The LLM generated 3 different phrasings of the same question. Searching with all of them found more relevant documents than using just the original query.

## HyDE (Hypothetical Document Embedding)

**What it does:** Generates a hypothetical answer to the query, then embeds that answer instead of the query.

**When to use it:** When queries are short questions but documents are long explanations. The hypothetical answer matches the style of documents better.

**Steps:**
1. Ask the LLM to write a short answer to the query.
2. Embed the hypothetical answer.
3. Search using that embedding.

In [ ]:
def hyde_search(query, top_k=3):
    response = client.chat.completions.create(
        model=MODEL, 
        max_tokens=100, 
        temperature=0,
        messages=[
            {
                "role": "user",
                "content": "Write a short technical "
                + f"paragraph answering: {query}"}]
    )
    
    hypothetical = response.choices[0].message.content
    print(f"Hypothetical rewritten Query:\n", hypothetical)

    hyde_embedding = get_embedding(hypothetical)
    return simple_search(hyde_embedding, top_k)

In [19]:
query = "What happens when CPU usage spikes?"

results = hyde_search(query)

print("results\n", results)

Hypothetical: When CPU usage spikes, the processor experiences a sudden increase in demand for computational resources, often due to resource-intensive applications or processes. This can lead to several outcomes: the CPU may prioritize tasks based on their scheduling algorithms, potentially causing some processes to be delayed or throttled. As a result, system responsiveness may decrease, leading to lag or stuttering in user interfaces and applications. Additionally, if the spike is sustained, thermal management mechanisms may activate, such as throttling the CPU speed to prevent overheating
results
 [({'id': 1, 'text': 'VPA adjusts CPU and memory requests. Changes resource requests, not replica count.', 'source': 'scaling', 'date': '2024-01'}, 0.3883985233342154), ({'id': 0, 'text': 'HPA scales Pods based on CPU. Default interval is 15 seconds. Supports custom metrics.', 'source': 'scaling', 'date': '2024-01'}, 0.34590079929312084), ({'id': 2, 'text': 'KEDA scales based on external e

**What happened:** The LLM wrote a hypothetical answer about CPU spikes and autoscaling. That hypothetical answer is closer in style to the actual documents, so embedding it produces better search results.

## Sub-Question Decomposition

**What it does:** Breaks a complex question into simpler sub-questions. Searches for each one separately.

**When to use it:** When the user asks a question that spans multiple topics. One search cannot cover all parts.

**Steps:**
1. Ask the LLM to split the question into 2-3 simpler parts.
2. Search for each sub-question.
3. Combine all retrieved documents.

In [21]:
def decompose_search(query):
    response = client.chat.completions.create(
        model=MODEL, 
        max_tokens=200, 
        temperature=0,
        response_format={
            "type": "json_object"
        },
        messages=[
            {
                "role": "system", 
                "content": "Return JSON only."
            },
            {
                "role": "user", 
                "content":
                "Break this question into 2-3 simpler "
                + f"sub-questions.\nQuestion: \"{query}\"\n"
                + 'Return: {"sub_questions": ["q1", "q2"]}'
            }
        ]
    )
    
    content = response.choices[0].message.content
    sub_questions = json.loads(content)["sub_questions"]
    
    return sub_questions

In [22]:
query = ("How do I set up autoscaling with "
         "monitoring and alerting?")

sub_qs = decompose_search(query)
for sq in sub_qs:
    print(f"Subquestion - {sq}")

Subquestion - What are the steps to set up autoscaling in my environment?
Subquestion - How can I implement monitoring for my autoscaling setup?
Subquestion - What methods can I use to configure alerting based on the monitoring data?


In [23]:
# Search for each sub-question
all_docs = {}

for sub_question in sub_qs:
    
    embedding = get_embedding(sub_question)
    
    results = simple_search(embedding, top_k=2)
    
    for i in range(len(results)):
        doc = results[i][0]
        score = results[i][1]
        
        doc_id = doc["id"]
        
        if doc_id not in all_docs:
            all_docs[doc_id] = (doc, score)
        elif score > all_docs[doc_id][1]:
            all_docs[doc_id] = (doc, score)

In [24]:
ranked = sorted(
    all_docs.values(),
    key=lambda x: x[1], reverse=True
)

print(ranked)

[({'id': 8, 'text': 'Prometheus scrapes /metrics endpoints. Grafana visualizes. AlertManager routes alerts.', 'source': 'monitoring', 'date': '2024-02'}, 0.47226639216946015), ({'id': 0, 'text': 'HPA scales Pods based on CPU. Default interval is 15 seconds. Supports custom metrics.', 'source': 'scaling', 'date': '2024-01'}, 0.43304645340158343), ({'id': 2, 'text': 'KEDA scales based on external events like queues. Supports 50+ sources. Scales to zero.', 'source': 'scaling', 'date': '2024-03'}, 0.33405047100140345)]


**What happened:** The complex question was split into simpler parts (autoscaling, monitoring, alerting). Searching for each part found documents across multiple topics that together answer the full question.

## Self-Query (LLM Generates Filters)

**What it does:** The LLM parses the user's query into a search query plus metadata filters.

**When to use it:** When users mix search terms with filter criteria. Example: "Show me security docs from March 2024."

**Steps:**
1. Ask the LLM to extract the search part and the filter part.
2. Apply the metadata filters to narrow the document set.
3. Run semantic search on the filtered set.

In [25]:
def self_query_search(query):
    response = client.chat.completions.create(
        model=MODEL, 
        max_tokens=200, 
        temperature=0,
        response_format={
            "type": "json_object"
        },
        messages=[
            {
                "role": "system", 
                "content": "Return JSON only."
            },
            {
                "role": "user", 
                "content":
                "Parse this query into a search query and "
                + "metadata filters.\n"
                + "Available fields: source (scaling, "
                + "networking, security, tooling, mesh, "
                + "monitoring, storage), date (YYYY-MM)\n"
                + f"Query: \"{query}\"\n"
                + 'Return: {"search_query": "...", '
                + '"filters": {"source": null, '
                + '"date_after": null}}'
            }
        ]
    )

    content = response.choices[0].message.content
    parsed = json.loads(content)

    print(parsed)
    
    return parsed

In [26]:
query = "What security features were added after Feb 2024?"

parsed = self_query_search(query)

{'search_query': 'security features added', 'filters': {'source': None, 'date_after': '2024-02'}}


In [29]:
# Apply filters and search
filtered = []
filters = parsed["filters"]

for doc in KNOWLEDGE_BASE:
    
    keep = True
    
    if filters.get("source"):
        if doc["source"] != filters["source"]:
            keep = False
    
    if filters.get("date_after"):
        if doc["date"] < filters["date_after"]:
            keep = False
    
    if keep:
        filtered.append(doc)

print("filtered \n", filtered)
print(f"Filtered to {len(filtered)} documents")

filtered 
 [{'id': 2, 'text': 'KEDA scales based on external events like queues. Supports 50+ sources. Scales to zero.', 'source': 'scaling', 'date': '2024-03'}, {'id': 3, 'text': 'Kubernetes Services: ClusterIP for internal, NodePort for external, LoadBalancer for cloud.', 'source': 'networking', 'date': '2024-02'}, {'id': 4, 'text': 'Network Policies restrict Pod-to-Pod traffic using label selectors. Default: all allowed.', 'source': 'security', 'date': '2024-02'}, {'id': 6, 'text': 'Helm charts template Kubernetes manifests. Supports dependencies, hooks, rollbacks.', 'source': 'tooling', 'date': '2024-03'}, {'id': 7, 'text': 'Istio provides mutual TLS, traffic splitting, circuit breaking, tracing with Jaeger.', 'source': 'mesh', 'date': '2024-04'}, {'id': 8, 'text': 'Prometheus scrapes /metrics endpoints. Grafana visualizes. AlertManager routes alerts.', 'source': 'monitoring', 'date': '2024-02'}]
Filtered to 6 documents


In [28]:
embedding = get_embedding(parsed["search_query"])

scores = []
for doc in filtered:
    score = cosine_similarity(
        embedding, 
        DOC_EMBEDDINGS[doc["id"]]
    )
    scores.append((doc, score))
scores.sort(key=lambda x: x[1], reverse=True)


print("\nResults:")
for i in range(min(3, len(scores))):
    doc = scores[i][0]
    score = scores[i][1]
    src = doc['source']
    date = doc['date']
    text_preview = doc['text'][:45]
    print(f"  {score:.3f} [{date}] [{src}]  {text_preview}...")


Results:
  0.276 [2024-04] [mesh]  Istio provides mutual TLS, traffic splitting,...
  0.220 [2024-03] [scaling]  KEDA scales based on external events like que...
  0.211 [2024-02] [security]  Network Policies restrict Pod-to-Pod traffic ...


**What happened:** The LLM extracted "security features" as the search query and "after February 2024" as a date filter. Only documents matching the filter were searched.

## Parent-Child Retrieval

**What it does:** Uses small chunks for precise matching, then returns the larger parent chunk that contains more context.

**When to use it:** When you need both precision (small chunks match better) and context (large chunks give the LLM more information).

**Steps:**
1. Create small child chunks and large parent chunks.
2. Search using the small chunks (more precise matching).
3. Return the parent chunk (more context for the LLM).

In [30]:
# Parent chunks (large, full context)
PARENTS = [
    {
        "id": "p0", 
        "text": KNOWLEDGE_BASE[0]["text"] + " " + KNOWLEDGE_BASE[1]["text"] + " " + KNOWLEDGE_BASE[2]["text"]
    },
    {
        "id": "p1", 
        "text": KNOWLEDGE_BASE[3]["text"] + " " + KNOWLEDGE_BASE[4]["text"]
    },
]

In [31]:
# Child chunks (small, precise)
CHILDREN = [
    {
        "text": "HPA scales Pods based on CPU.",
        "parent_id": "p0"
    },
    {
        "text": "VPA adjusts CPU and memory requests.",
        "parent_id": "p0"
    },
    {
        "text": "KEDA scales based on external events.",
        "parent_id": "p0"
    },
    {
        "text": "ClusterIP is internal. NodePort external.",
        "parent_id": "p1"
    },
    {
        "text": "Network Policies restrict Pod traffic.",
        "parent_id": "p1"
    },
]

child_texts = []
for child in CHILDREN:
    child_texts.append(child["text"])


child_embeddings = get_embeddings(child_texts)
child_embeddings = np.array(child_embeddings, dtype="float32")

In [32]:
query = "How does CPU-based scaling work?"
query_embedding = get_embedding(query)


scores = []
for i in range(len(CHILDREN)):
    score = cosine_similarity(
        query_embedding, 
        child_embeddings[i]
    )
    scores.append((i, score))
scores.sort(key=lambda x: x[1], reverse=True)

print("scores: ", scores)


best_index = scores[0][0]
best_child = CHILDREN[best_index]

print(best_index)
print(best_child)


scores:  [(0, 0.5662320822446323), (1, 0.44941988350830625), (2, 0.310768404595133), (3, 0.2267963858181096), (4, 0.19216459870448685)]
0
{'text': 'HPA scales Pods based on CPU.', 'parent_id': 'p0'}


In [33]:
parent = None
for p in PARENTS:
    if p["id"] == best_child["parent_id"]:
        parent = p
        break


print(query)
print(best_child)
print(parent)

How does CPU-based scaling work?
{'text': 'HPA scales Pods based on CPU.', 'parent_id': 'p0'}
{'id': 'p0', 'text': 'HPA scales Pods based on CPU. Default interval is 15 seconds. Supports custom metrics. VPA adjusts CPU and memory requests. Changes resource requests, not replica count. KEDA scales based on external events like queues. Supports 50+ sources. Scales to zero.'}


**What happened:** The small child chunk "HPA scales Pods based on CPU" matched the query precisely. Instead of returning just that small chunk, we returned the full parent chunk with all the scaling information. The LLM now has more context to generate a complete answer.

## Contextual Compression

**What it does:** After retrieving documents, extracts only the parts relevant to the query. Removes noise.

**When to use it:** When retrieved documents are long and contain information unrelated to the query.

**Steps:**
1. Retrieve the top documents.
2. For each document, ask the LLM to extract only the relevant parts.
3. Send the compressed text to the final LLM for answer generation.

In [36]:
def compressed_search(query, top_k=3):
    
    embedding = get_embedding(query)
    results = simple_search(embedding, top_k=top_k)

    compressed = []
    for i in range(len(results)):
        doc = results[i][0]
        score = results[i][1]
        
        response = client.chat.completions.create(
            model=MODEL, 
            max_tokens=80, 
            temperature=0,
            messages=[
                {
                    "role": "user", 
                    "content":
                    "Extract ONLY the parts relevant to: "
                    + f"{query}\nDocument: {doc['text']}"
                    + "\nIf nothing relevant, say NOT_RELEVANT."}
                ]
        )

        extract = response.choices[0].message.content
        print("extract:\n", extract)
        
        if "NOT_RELEVANT" not in extract:
            compressed.append((doc, extract, score))
    
    return compressed

In [38]:
query = "How does scaling to zero work?"

results = compressed_search(query)

print(results)

extract:
 KEDA scales to zero.
extract:
 NOT_RELEVANT
extract:
 NOT_RELEVANT
[({'id': 2, 'text': 'KEDA scales based on external events like queues. Supports 50+ sources. Scales to zero.', 'source': 'scaling', 'date': '2024-03'}, 'KEDA scales to zero.', 0.42085229377491223)]


**What happened:** Each retrieved document was compressed to keep only the parts about scaling to zero. The LLM for final answer generation now gets focused, relevant text instead of long documents with mixed content.

## Summary

- **Multi-query** generates different phrasings of the query to find more results.
- **HyDE** generates a hypothetical answer and embeds that instead of the short query.
- **Sub-question decomposition** splits complex questions into simpler parts and searches for each.
- **Self-query** lets the LLM extract metadata filters from the user's natural language query.
- **Parent-child retrieval** matches on small chunks for precision but returns large chunks for context.
- **Contextual compression** removes irrelevant parts from retrieved documents before sending to the LLM.

**Cost ranking (cheapest to most expensive):**
1. Parent-child: free (just index lookup)
2. Multi-query: 1 LLM call + N embedding calls
3. HyDE: 1 LLM call + 1 embedding call
4. Self-query: 1 LLM call
5. Sub-question: 1 LLM call + N embedding calls
6. Compression: N LLM calls (one per retrieved doc)